Imports


In [25]:
import pandas as pd

# Dataset de Online Retail


Según la página desde donde obtenemos el dataset, el mismo contiene lo siguiente:

"Este es un conjunto de datos transaccionales que contiene todas las transacciones que ocurrieron entre el 01/12/2010 y el 09/12/2011 para una tienda minorista en línea no comercial registrada y con sede en el Reino Unido. La empresa vende principalmente regalos únicos para todas las ocasiones. Muchos clientes de la empresa son mayoristas."


Y la descripción de sus columnas es:

| Nombre de Variable | Rol     | Tipo       | Descripción                                                                                                       | Unidades | Valores Faltantes |
| ------------------ | ------- | ---------- | ----------------------------------------------------------------------------------------------------------------- | -------- | ----------------- |
| InvoiceNo          | ID      | Categórica | Número entero de 6 dígitos asignado de forma única a cada transacción. Si empieza con "C", indica una cancelación | -        | no                |
| StockCode          | ID      | Categórica | Número entero de 5 dígitos asignado de forma única a cada producto                                                | -        | no                |
| Description        | Feature | Categórica | Nombre del producto                                                                                               | -        | no                |
| Quantity           | Feature | Entera     | Cantidad de cada producto (ítem) por transacción                                                                  | -        | no                |
| InvoiceDate        | Feature | Fecha      | Fecha y hora en que se generó cada transacción                                                                    | -        | no                |
| UnitPrice          | Feature | Continua   | Precio por unidad del producto                                                                                    | Libras   | no                |
| CustomerID         | Feature | Categórica | Número entero de 5 dígitos asignado de forma única a cada cliente                                                 | -        | no                |
| Country            | Feature | Categórica | Nombre del país de residencia de cada cliente                                                                     | -        | no                |


Cargamos el dataset


In [69]:
df = pd.read_excel("data/Online Retail.xlsx", sheet_name=0)

Le damos una primera mirada


In [70]:
df.head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


A primera vista vemos que hay unas **541909** filas en este dataset. Tenemos alrededor de mil registros con "Description" nula y 100mil registros con "CustomerID" nulo.


Vemos más información de los campos


In [72]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Quantity,541909.0,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303


Observamos varias cosas de este output.


1. El valor mínimo y máximo del campo "InvoiceDate" confirman el rango de fechas que maneja el dataset, según la información que nos brindaba la página cuando lo descargamos.


2. El campo "Quantity" tiene un valor mínimo que es negativo. Lo cual a primera vista no tiene sentido. Vamos a tener que investigar más.


3. El campo "UnitPrice" también tiene un valor mínimo negativo. De nuevo, vamos a tener que investigar más al respecto.


4. El campo "Quantity" y el campo "UnitPrice" tienen hasta el percentil 75 valores razonables, como ser una "Quantity" de 10 y un "UnitPrice" de 4,13. Pero el valor máximo de ambos son demasiado altos, y casi seguro que se trata de outliers.


In [73]:
df.describe(include="object").T

,count,unique,top,freq
InvoiceNo,541909,25900,573585,1114
StockCode,541909,4070,85123A,2313
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369
Country,541909,38,United Kingdom,495478


Observamos varias cosas de este output.


1. Hay 25900 valores de "InvoiceNumber" únicos, por lo que las 541909 filas del dataset son líneas de venta de un total de 25900 facturas o compras.


2. El número de "StockCode" y de "Description" no coinciden, lo cual es extraño porque a simple vista uno podría asumir que cada stock se corresponde con una descripción de un producto.


3. El país "United Kingdom" es por lejos el más frecuente en todo el dataset, con un total de 495478 filas en las que figura.


In [74]:
# Vemos algunos de los registros sin CustomerID
df[df["CustomerID"].isna()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-12-01 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-12-01 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-12-01 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-12-01 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-12-01 14:32:00,0.43,NaN,United Kingdom


Hay algunos registros inusuales, como el primero que tiene un "UnitPrice" de 0,00. Pero también hay registros que parecen legítimos, con descripciones de producto, cantidades válidades y precios razonables.


Tendremos que ver más adelante qué hacer con estos registros sin "CustomerID", puede que tengamos que descartarlos, o puede que podamos rescatar algunos.


In [75]:
# Revisamos si hay algún registro sin "CustomerID" que tenga un "InvoiceNo"
df[df["CustomerID"].isna() & df["InvoiceNo"].notna()].count()

InvoiceNo      135080
StockCode      135080
Description    133626
Quantity       135080
InvoiceDate    135080
UnitPrice      135080
CustomerID          0
Country        135080
dtype: int64

Tenemos 135080 filas que tienen un "InvoiceNo" y NO tienen un "CustomerId". Vamos a tratar de buscar otras filas con el mismo "InvoiceNo", y ver si podemos rescatar el "CustomerID" de ese modo.


In [76]:
# Guardamos los "InvoiceNo" que tienen filas sin "CustomerID"
invoices_sin_customer = df[df["CustomerID"].isna() & df["InvoiceNo"].notna()][
    "InvoiceNo"
].unique()
print(f"Número de facturas únicas sin CustomerID: {len(invoices_sin_customer)}")

# Guardamos los "InvoiceNo" que tienen filas con "CustomerID"
invoices_con_customer = df[df["CustomerID"].notna()]["InvoiceNo"].unique()

# Intersectamos los dos conjuntos de "InvoiceNo"
# para ver cuántas facturas tienen tanto filas con CustomerID como sin CustomerID
invoices_recuperables = set(invoices_sin_customer) & set(invoices_con_customer)
print(
    f"Facturas que tienen tanto filas con CustomerID como sin CustomerID: {len(invoices_recuperables)}"
)

Número de facturas únicas sin CustomerID: 3710
Facturas que tienen tanto filas con CustomerID como sin CustomerID: 0


Parece que la estrategia de recuperar "CustomerID" usando otras filas de la misma factura no va a ser posible. No hay ningún caso en el que esto suceda.


In [77]:
# Vemos los registros con cantidades negativas
df[df["Quantity"] < 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


Para esta muestra de 10 que tomamos, podemos ver que todos estos registros con cantidad negativa tienen un "CustomerID" asociado, y además, el "InvoiceNo" de todos empieza con la letra C.


Revisando la celda de arriba con la descripción de cada columna, vemos que la letra C se traduce como una transacción cancelada. Capaz eso explica la cantidad negativa.


In [78]:
# Casteamos "InvoiceNo" a string para poder analizar mejor su contenido
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["InvoiceNo"].head(5)

0    536365
1    536365
2    536365
3    536365
4    536365
Name: InvoiceNo, dtype: object

In [79]:
# Usando una expresión regular, revisamos si hay algún registro que
# no siga el patrón de 6 dígitos con un prefijo opcional de C en "InvoiceNo"
df[~df["InvoiceNo"].str.match(r"^(C)?\d{6}$")].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


Vemos estos 3 registros con una "Description" que da a entender que se trata de ajustes de deuda incobrable. Podría significar que fueron transacciones hechas para reflejar esa pérdida en el sistema.


No tienen importancia para nuestro análisis, pues no representan compras como tal, por lo que vamos a eliminarlos más adelante.


In [80]:
# Casteamos "StockCode" a string para poder analizar mejor su contenido
df["StockCode"] = df["StockCode"].astype(str)

In [81]:
# Usando una expresión regular, revisamos si hay algún registro que
# no siga el patrón de 5 dígitos en "StockCode"
df[~df["StockCode"].str.match(r"^\d{5}$")].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France
49,536373,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:02:00,2.55,17850.0,United Kingdom
51,536373,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 09:02:00,2.75,17850.0,United Kingdom
60,536373,82494L,WOODEN FRAME ANTIQUE WHITE,6,2010-12-01 09:02:00,2.55,17850.0,United Kingdom
61,536373,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 09:02:00,3.39,17850.0,United Kingdom
62,536373,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 09:02:00,3.39,17850.0,United Kingdom


Vemos que una gran cantidad de registros no responden al formato que se especifica en la descripción de la columna "StockCode". Unas **54873** filas.


Parecen transacciones válidas a primera vista. Con un "CustomerID" asociado, "UnitPrice" positivo y "Quantity" también positiva. Al menos eso se observa en la muestra de 10 que visualizamos.


In [82]:
# Eliminamos los números iniciales de "StockCode" seguido de ninguna, una o varias
# letras, para ir reduciendo el número de posibles valores únicos
df[~df["StockCode"].str.match(r"^\d{5}[A-Z]*$")]["StockCode"].unique()

array(['POST', 'D', 'C2', 'DOT', 'M', 'BANK CHARGES', '15056bl', '15056p',
       '47566b', '72349b', '84872a', '84970l', '84970s', '85039a',
       '18098c', '84509c', '84534b', '84559a', '84596e', '84884a',
       '85014a', '85014b', '85036b', '72803b', '47559b', '47591d',
       '48173c', '82494l', '84030e', '84997b', '84997c', '84997d',
       '85049a', '85049e', '85099f', '85123a', 'S', 'AMAZONFEE', '84989a',
       '15056n', '84559b', '84660c', '85114c', '72351a', '84510c',
       '84596b', '85035c', '85132a', '47570b', '84558a', 'DCGS0076',
       '35809a', '79066k', '84536b', '84968a', '84510a', '84968e',
       '85035a', '84660b', 'DCGS0003', '84796a', '72801d', '85114a',
       'gift_0001_40', 'DCGS0070', '85231b', '85114b', '85160a', 'm',
       '82613b', '82613c', '84795b', '47518f', '85035b', '84968f',
       '85049g', 'gift_0001_50', 'gift_0001_30', 'gift_0001_20', '85231g',
       '85040a', '47590b', '72807a', '72801c', '84997a', 'DCGS0055',
       'DCGS0072', 'DCGS0074'

Tal parece que las letras también pueden estar en minúscula, por lo que agregamos eso a nuestra expresión regular y volvemos a intentar


In [83]:
df[~df["StockCode"].str.match(r"^\d{5}[a-zA-Z]*$")]["StockCode"].unique()

array(['POST', 'D', 'C2', 'DOT', 'M', 'BANK CHARGES', 'S', 'AMAZONFEE',
       'DCGS0076', 'DCGS0003', 'gift_0001_40', 'DCGS0070', 'm',
       'gift_0001_50', 'gift_0001_30', 'gift_0001_20', 'DCGS0055',
       'DCGS0072', 'DCGS0074', 'DCGS0069', 'DCGS0057', 'DCGSSBOY',
       'DCGSSGIRL', 'gift_0001_10', 'PADS', 'DCGS0004', 'DCGS0073',
       'DCGS0071', 'DCGS0068', 'DCGS0067', 'DCGS0066P', 'B', 'CRUK'],
      dtype=object)

Hay muchas posibilidades. Vamos a tener que revisar uno por uno estos "StockCode" para ver si son válidos


> **Note:** El resto del análisis del campo StockCode se realizó en una notebook separada para mantener el EDA más conciso. Dicho análisis se encuentra en la carpeta `extra_analysis`.


Luego de realizar el análisis de los diferentes StockCode en el otro notebook. Las decisiones a tomar son las siguientes:


- POST: representa costos de envío. Lo vamos a excluir.
- D: representa descuentos. Lo vamos a excluir.
- C2: represneta costos de envío. Lo vamos a excluir.
- DOT: representa costos de envío. Lo vamos a excluir.
- M: representa cargas manuales. Lo vamos a excluir.
- BANK CHARGES: representa costos bancarios. Lo vamos a excluir.
- S: representa muestras gratis. Lo vamos a excluir.
- AMAZONFEE: representa tarifas de Amazon por ventas en plataforma. Lo vamos a excluir.
- ^DCGS: no sabemos qué representan. Lo vamos a excluir.
- ^gift: representa regalos. Lo vamos a excluir.
- m: representa cargas manuales. Lo vamos a excluir.
- PADS: representa compras de almohadillas. Lo vamos a CONSERVAR.
- B: representa ajustes contables. Lo vamos a excluir.
- CRUK: representa comisinones a una organización benéfica. Lo vamos a excluir.
